In [4]:
!pip install -q numba scipy

import numpy as np
from numba import cuda, float32
import time

print("CUDA available:", cuda.is_available())
if cuda.is_available():
    print("GPU:", cuda.get_current_device().name)
else:
    raise RuntimeError("CUDA GPU is not available. In Colab select Runtime > Change runtime type > GPU.")


CUDA available: True
GPU: Tesla T4


In [5]:
@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a[i] + b[i]

N = 1000
a = np.arange(N, dtype=np.float32)
b = (np.arange(N, dtype=np.float32) * 2)
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_out = cuda.device_array_like(a)

threads = 256
blocks = (N + threads - 1) // threads
vector_add_1d[blocks, threads](d_a, d_b, d_out)
result = d_out.copy_to_host()
np.testing.assert_allclose(result, a + b, atol=1e-5)
print("Task 1: PASSED", result[:10])


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Task 1: PASSED [ 0.  3.  6.  9. 12. 15. 18. 21. 24. 27.]


In [6]:

@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)

    while i < x.size:
        out[i] = x[i] * alpha
        i += stride


# Create an array
N = 10000
x = np.arange(N, dtype=np.float32)
alpha = np.float32(2.5)
out = np.zeros(N, dtype=np.float32)

# Copy arrays to GPU
d_x = cuda.to_device(x)
d_out = cuda.to_device(out)

# Run the kernel
threads = 256
blocks = 64

vector_scale_grid_stride[blocks, threads](d_x, alpha, d_out)

# Copy result back to CPU
result = d_out.copy_to_host()

# Check result
np.testing.assert_allclose(result, x * alpha, atol=1e-5)

print("Task 2: PASSED")
print("First 10 results:", result[:10])


Task 2: PASSED
First 10 results: [ 0.   2.5  5.   7.5 10.  12.5 15.  17.5 20.  22.5]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [7]:

@cuda.jit
def matrix_add_2d(A, B, C):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x

    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]


# Create two matrices
M = 100
N = 150

A = np.ones((M, N), dtype=np.float32)
B = np.ones((M, N), dtype=np.float32) * 2
C = np.zeros((M, N), dtype=np.float32)

# Copy matrices to GPU
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)

# Configure CUDA threads and blocks
threads = (16, 16)
blocks = ((N + 15) // 16, (M + 15) // 16)

# Run the kernel
matrix_add_2d[blocks, threads](d_A, d_B, d_C)

# Copy result back to CPU
result = d_C.copy_to_host()

# Check result
np.testing.assert_allclose(result, A + B, atol=1e-5)

print("Task 3: PASSED")
print("Matrix shape:", result.shape)
print("First row:", result[0, :10])


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 70 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Task 3: PASSED
Matrix shape: (100, 150)
First row: [3. 3. 3. 3. 3. 3. 3. 3. 3. 3.]


In [8]:

@cuda.jit
def matrix_transpose_naive(A, B):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x

    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]


# Create matrix
M = 4
N = 6

A = np.arange(M * N, dtype=np.float32).reshape(M, N)
B = np.zeros((N, M), dtype=np.float32)

# Copy to GPU
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)

# Configure blocks and threads
threads = (16, 16)
blocks = ((N + 15) // 16, (M + 15) // 16)

# Run kernel
matrix_transpose_naive[blocks, threads](d_A, d_B)

# Copy result back
result = d_B.copy_to_host()

# Check result
np.testing.assert_allclose(result, A.T, atol=1e-5)

print("Task 4: PASSED")
print("Original matrix:")
print(A)
print("Transposed matrix:")
print(result)


Task 4: PASSED
Original matrix:
[[ 0.  1.  2.  3.  4.  5.]
 [ 6.  7.  8.  9. 10. 11.]
 [12. 13. 14. 15. 16. 17.]
 [18. 19. 20. 21. 22. 23.]]
Transposed matrix:
[[ 0.  6. 12. 18.]
 [ 1.  7. 13. 19.]
 [ 2.  8. 14. 20.]
 [ 3.  9. 15. 21.]
 [ 4. 10. 16. 22.]
 [ 5. 11. 17. 23.]]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [9]:

@cuda.jit
def volume_init_3d(vol, D, H, W):
    x = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    y = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    z = cuda.blockIdx.z * cuda.blockDim.z + cuda.threadIdx.z

    if z < D and y < H and x < W:
        vol[z, y, x] = z * 10000 + y * 100 + x


# Create a 3D array
D = 4
H = 5
W = 6

vol = np.zeros((D, H, W), dtype=np.int32)

# Copy array to GPU
d_vol = cuda.to_device(vol)

# Configure CUDA blocks and threads
threads = (8, 8, 8)
blocks = (
    (W + 7) // 8,
    (H + 7) // 8,
    (D + 7) // 8
)

# Run kernel
volume_init_3d[blocks, threads](d_vol, D, H, W)

# Copy result back to CPU
result = d_vol.copy_to_host()

# Check result
z, y, x = np.indices((D, H, W))
expected = z * 10000 + y * 100 + x

np.testing.assert_array_equal(result, expected)

print("Task 5: PASSED")
print("Array shape:", result.shape)
print("First layer:")
print(result[0])
print("Second layer, first row:", result[1, 0])


Task 5: PASSED
Array shape: (4, 5, 6)
First layer:
[[  0   1   2   3   4   5]
 [100 101 102 103 104 105]
 [200 201 202 203 204 205]
 [300 301 302 303 304 305]
 [400 401 402 403 404 405]]
Second layer, first row: [10000 10001 10002 10003 10004 10005]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [10]:

@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    y, x = cuda.grid(2)

    if y < img_rgb.shape[0] and x < img_rgb.shape[1]:
        r = img_rgb[y, x, 0]
        g = img_rgb[y, x, 1]
        b = img_rgb[y, x, 2]

        img_gray[y, x] = 0.299 * r + 0.587 * g + 0.114 * b


# Create a test RGB image
H = 100
W = 100

img_rgb = np.random.randint(
    0, 256, (H, W, 3), dtype=np.uint8
)

img_gray = np.zeros((H, W), dtype=np.float32)

# Copy data to GPU
d_rgb = cuda.to_device(img_rgb)
d_gray = cuda.to_device(img_gray)

# Configure CUDA
threads = (16, 16)
blocks = (
    (H + 15) // 16,
    (W + 15) // 16
)

# Run kernel
rgb_to_grayscale[blocks, threads](d_rgb, d_gray)

# Copy result back to CPU
result = d_gray.copy_to_host()

# Check result
expected = (
    0.299 * img_rgb[:, :, 0].astype(np.float32)
    + 0.587 * img_rgb[:, :, 1].astype(np.float32)
    + 0.114 * img_rgb[:, :, 2].astype(np.float32)
)

np.testing.assert_allclose(result, expected, atol=1e-4)

print("Task 6: PASSED")
print("Image shape:", img_rgb.shape)
print("Grayscale shape:", result.shape)
print("First 5 pixels:", result[0, :5])


Task 6: PASSED
Image shape: (100, 100, 3)
Grayscale shape: (100, 100)
First 5 pixels: [ 91.112 142.471  60.665 232.29   73.835]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 49 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [11]:

@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)

    if i < out.size:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]


# Create arrays
N = 10000

a = np.ones(N, dtype=np.float32) * 10
b = np.ones(N, dtype=np.float32) * 20
t = np.ones(N, dtype=np.float32) * 0.5

out = np.zeros(N, dtype=np.float32)

# Copy arrays to GPU
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_t = cuda.to_device(t)
d_out = cuda.to_device(out)

# Configure CUDA
threads = 256
blocks = (N + threads - 1) // threads

# Run kernel
vector_lerp[blocks, threads](d_a, d_b, d_t, d_out)

# Copy result back to CPU
result = d_out.copy_to_host()

# Check result
expected = (1 - t) * a + t * b
np.testing.assert_allclose(result, expected, atol=1e-5)

print("Task 7: PASSED")
print("First 10 results:", result[:10])


Task 7: PASSED
First 10 results: [15. 15. 15. 15. 15. 15. 15. 15. 15. 15.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [12]:

@cuda.jit
def vector_threshold(x, thresh, mask):
    i = cuda.grid(1)

    if i < x.size:
        if x[i] >= thresh:
            mask[i] = 1
        else:
            mask[i] = 0


# Create an array
N = 10000
x = np.arange(N, dtype=np.float32)
thresh = np.float32(5000)

mask = np.zeros(N, dtype=np.uint8)

# Copy data to GPU
d_x = cuda.to_device(x)
d_mask = cuda.to_device(mask)

# Configure CUDA
threads = 256
blocks = (N + threads - 1) // threads

# Run kernel
vector_threshold[blocks, threads](d_x, thresh, d_mask)

# Copy result back to CPU
result = d_mask.copy_to_host()

# Check result
expected = (x >= thresh).astype(np.uint8)
np.testing.assert_array_equal(result, expected)

print("Task 8: PASSED")
print("First 10 mask values:", result[:10])
print("Number of elements above threshold:", np.sum(result))


Task 8: PASSED
First 10 mask values: [0 0 0 0 0 0 0 0 0 0]
Number of elements above threshold: 5000


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [13]:

@cuda.jit
def copy_coalesced(in_arr, out_arr):
    i = cuda.grid(1)

    if i < in_arr.size:
        out_arr[i] = in_arr[i]


@cuda.jit
def copy_strided(in_arr, out_arr, stride):
    i = cuda.grid(1)

    if i < in_arr.size:
        j = (i * stride) % in_arr.size
        out_arr[i] = in_arr[j]


def benchmark_strided_access(in_arr, out_arr, stride):
    d_in = cuda.to_device(in_arr)
    d_out = cuda.device_array_like(out_arr)

    threads = 256
    blocks = (in_arr.size + threads - 1) // threads

    # Select kernel
    if stride == 1:
        kernel = copy_coalesced
        args = (d_in, d_out)
    else:
        kernel = copy_strided
        args = (d_in, d_out, stride)

    # Warm-up
    kernel[blocks, threads](*args)
    cuda.synchronize()

    # Measure execution time
    start = cuda.event()
    end = cuda.event()

    start.record()
    for _ in range(20):
        kernel[blocks, threads](*args)
    end.record()
    end.synchronize()

    elapsed_ms = cuda.event_elapsed_time(start, end) / 20

    result = d_out.copy_to_host()

    return elapsed_ms, result


# Create test data
N = 1_000_000
data = np.arange(N, dtype=np.float32)
output = np.empty_like(data)

# Test different strides
for stride in [1, 2, 4, 8, 16, 32]:
    elapsed, result = benchmark_strided_access(
        data, output, stride
    )

    expected = data if stride == 1 else data[
        (np.arange(N) * stride) % N
    ]

    np.testing.assert_array_equal(result, expected)

    print(f"Stride {stride:2d}: {elapsed:.4f} ms")

print("Task 9: PASSED")


Stride  1: 0.0476 ms
Stride  2: 0.0588 ms
Stride  4: 0.0814 ms
Stride  8: 0.1267 ms
Stride 16: 0.1199 ms
Stride 32: 0.1058 ms
Task 9: PASSED
